# 📊 ĐÁNH GIÁ CHẤT LƯỢNG DỮ LIỆU AI SINH RA VS DỮ LIỆU THỰC TẾ (INPUT_TURN2_VONG1)

Notebook này phân tích và so sánh chất lượng, độ dài, phân bổ thực thể y khoa (chẩn đoán, thuốc, triệu chứng) giữa:
1. **Dữ liệu thực tế gốc (`input_turn2_vong1/input`)**
2. **Dữ liệu AI sinh ra bởi bộ của Long (`sample_Long`)**
3. **Dữ liệu AI sinh ra bởi bộ của A (`sample_A`)**

In [ ]:
import os
import json
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Đường dẫn các thư mục dữ liệu
BASE_DIR = os.path.abspath(os.getcwd())
REF_DIR = os.path.join(BASE_DIR, "input_turn2_vong1", "input")
GEN_A_DIR = os.path.join(BASE_DIR, "sample_A")
GEN_LONG_DIR = os.path.join(BASE_DIR, "sample_Long")

print(f"Thư mục gốc: {BASE_DIR}")
print(f"Thư mục dữ liệu gốc (100 files): {REF_DIR} - Tồn tại: {os.path.exists(REF_DIR)}")
print(f"Thư mục AI sample_A: {GEN_A_DIR} - Tồn tại: {os.path.exists(GEN_A_DIR)}")
print(f"Thư mục AI sample_Long: {GEN_LONG_DIR} - Tồn tại: {os.path.exists(GEN_LONG_DIR)}")

## 1. So sánh Số lượng Từ (Word Count) và Độ Dài Văn Bản
Văn bản lâm sàng thực tế thường có độ dài dao động từ 150 - 800 từ. Hãy xem phân bố độ dài của dữ liệu AI tạo ra có tương đồng hay không.

In [ ]:
def get_word_counts(directory):
    counts = []
    for root, _, files in os.walk(directory):
        for f in files:
            if f.endswith('.txt'):
                with open(os.path.join(root, f), 'r', encoding='utf-8') as file:
                    text = file.read()
                    words = text.split()
                    counts.append(len(words))
    return counts

ref_words = get_word_counts(REF_DIR)
gen_a_words = get_word_counts(os.path.join(GEN_A_DIR, 'input'))
gen_long_words = get_word_counts(os.path.join(GEN_LONG_DIR, 'input'))

print("=== PHÂN TÍCH ĐỘ DÀI TỪ (WORD COUNT) ===")
print(f"Dữ liệu Gốc (Turn 2):  Trung bình = {np.mean(ref_words):.1f} từ | Min = {np.min(ref_words)} | Max = {np.max(ref_words)}")
if gen_a_words:
    print(f"Dữ liệu AI (sample_A): Trung bình = {np.mean(gen_a_words):.1f} từ | Min = {np.min(gen_a_words)} | Max = {np.max(gen_a_words)}")
if gen_long_words:
    print(f"Dữ liệu AI (sample_Long): Trung bình = {np.mean(gen_long_words):.1f} từ | Min = {np.min(gen_long_words)} | Max = {np.max(gen_long_words)}")

# Vẽ biểu đồ so sánh cho cả 3 bộ dữ liệu
plt.figure(figsize=(10, 5))
plt.hist(ref_words, bins=20, alpha=0.4, label='Dữ liệu Gốc (Turn 2)', color='blue', density=True)
if gen_a_words:
    plt.hist(gen_a_words, bins=20, alpha=0.4, label='Dữ liệu AI (sample_A)', color='green', density=True)
if gen_long_words:
    plt.hist(gen_long_words, bins=20, alpha=0.4, label='Dữ liệu AI (sample_Long)', color='orange', density=True)
plt.title('So sánh phân bố độ dài văn bản (Word Count Density)')
plt.xlabel('Số từ')
plt.ylabel('Mật độ')
plt.legend()
plt.show()

## 2. So sánh Phân Bố và Số Lượng Thực Thể (Entities Distribution)
Mỗi mẫu dữ liệu cần có các thực thể lâm sàng gán nhãn: `CHẨN_ĐOÁN`, `THUỐC`, `TRIỆU_CHỨNG`. Hãy cùng đếm số lượng thực thể trung bình trong mỗi file JSON.

In [ ]:
def analyze_entities(output_directory):
    entity_types = {}
    entities_per_file = []
    for root, _, files in os.walk(output_directory):
        for f in files:
            if f.endswith('.json') and f != 'stats.json':
                fp = os.path.join(root, f)
                try:
                    with open(fp, 'r', encoding='utf-8') as file:
                        data = json.load(file)
                        entities_per_file.append(len(data))
                        for item in data:
                            etype = item.get('type', 'UNKNOWN')
                            entity_types[etype] = entity_types.get(etype, 0) + 1
                except Exception:
                    pass
    return entity_types, entities_per_file

gen_a_types, gen_a_counts = analyze_entities(os.path.join(GEN_A_DIR, 'output'))
gen_long_types, gen_long_counts = analyze_entities(os.path.join(GEN_LONG_DIR, 'output'))

print("=== THỐNG KÊ THỰC THỂ GÁN NHÃN ===")
if gen_a_counts:
    print(f"[sample_A] Số lượng thực thể trung bình mỗi file: {np.mean(gen_a_counts):.1f} nhãn (Min={np.min(gen_a_counts)}, Max={np.max(gen_a_counts)})")
    print(f"[sample_A] Phân loại nhãn:")
    for t, cnt in sorted(gen_a_types.items()):
        print(f"   - {t:15}: {cnt:5d} nhãn ({cnt/sum(gen_a_types.values())*100:.1f}%)")

if gen_long_counts:
    print(f"\n[sample_Long] Số lượng thực thể trung bình mỗi file: {np.mean(gen_long_counts):.1f} nhãn (Min={np.min(gen_long_counts)}, Max={np.max(gen_long_counts)})")
    print(f"[sample_Long] Phân loại nhãn:")
    for t, cnt in sorted(gen_long_types.items()):
        print(f"   - {t:15}: {cnt:5d} nhãn ({cnt/sum(gen_long_types.values())*100:.1f}%)")

## 3. Đánh Giá Độ Đa Dạng Từ Vựng (Vocabulary Diversity)
Chúng ta dùng tỷ lệ Type-Token Ratio (TTR = Số từ độc bản / Tổng số từ) để xem dữ liệu AI sinh ra có bị lặp từ nhiều hay phong phú từ vựng tự nhiên như văn bản lâm sàng thực tế.

In [ ]:
def calculate_ttr(directory):
    all_words = []
    for root, _, files in os.walk(directory):
        for f in files:
            if f.endswith('.txt'):
                with open(os.path.join(root, f), 'r', encoding='utf-8') as file:
                    text = file.read().lower()
                    # Loại bỏ dấu câu cơ bản
                    text = re.sub(r'[^\w\s]', '', text)
                    all_words.extend(text.split())
    if not all_words:
        return 0
    unique_words = set(all_words)
    ttr = len(unique_words) / len(all_words)
    return ttr, len(all_words), len(unique_words)

ref_ttr, ref_tot, ref_uniq = calculate_ttr(REF_DIR)
a_ttr, a_tot, a_uniq = calculate_ttr(os.path.join(GEN_A_DIR, 'input'))
long_ttr, long_tot, long_uniq = calculate_ttr(os.path.join(GEN_LONG_DIR, 'input'))

print("=== TỶ LỆ ĐA DẠNG TỪ VỰNG (TYPE-TOKEN RATIO) ===")
print(f"Dữ liệu Gốc:  TTR = {ref_ttr*100:.2f}% | Tổng số từ = {ref_tot} | Từ độc nhất = {ref_uniq}")
if a_tot:
    print(f"sample_A:     TTR = {a_ttr*100:.2f}% | Tổng số từ = {a_tot} | Từ độc nhất = {a_uniq}")
if long_tot:
    print(f"sample_Long:  TTR = {long_ttr*100:.2f}% | Tổng số từ = {long_tot} | Từ độc nhất = {long_uniq}")

print("\n💡 Nhận xét: Chỉ số TTR càng gần nhau chứng tỏ độ đa dạng ngôn ngữ của AI càng tiệm cận dữ liệu thực tế.")

## 4. Kiểm Tra Tỷ Lệ Align Vị Trí (Position Alignment Health Check)
Nếu AI gán nhãn sai tọa độ `position` [start, end] so với văn bản gốc `.txt` thì dữ liệu đó sẽ bị lỗi khi train. Đoạn mã dưới đây đếm số lượng lỗi lệch vị trí trong thư mục output.

In [ ]:
def check_alignment_errors(member_dir):
    input_dir = os.path.join(member_dir, 'input')
    output_dir = os.path.join(member_dir, 'output')
    
    total_checked = 0
    errors = 0
    
    for root, _, files in os.walk(output_dir):
        for f in files:
            if f.endswith('.json') and f != 'stats.json':
                rel_p = os.path.relpath(os.path.join(root, f), output_dir)
                txt_p = os.path.join(input_dir, os.path.splitext(rel_p)[0] + '.txt')
                json_p = os.path.join(root, f)
                
                if os.path.exists(txt_p):
                    with open(txt_p, 'r', encoding='utf-8') as tf:
                        txt_content = tf.read()
                    with open(json_p, 'r', encoding='utf-8') as jf:
                        labels = json.load(jf)
                        
                    for item in labels:
                        total_checked += 1
                        start, end = item['position']
                        label_text = item['text']
                        sliced_text = txt_content[start:end]
                        
                        if sliced_text != label_text:
                            errors += 1
                            
    return errors, total_checked

errors_a, checked_a = check_alignment_errors(GEN_A_DIR)
errors_long, checked_long = check_alignment_errors(GEN_LONG_DIR)

print("=== KIỂM TRA SỨC KHỎE POSITION ALIGNMENT ===")
print(f"[sample_A]    Lệch tọa độ: {errors_a} / {checked_a} nhãn ({errors_a/checked_a*100 if checked_a else 0:.3f}% lỗi)")
print(f"[sample_Long] Lệch tọa độ: {errors_long} / {checked_long} nhãn ({errors_long/checked_long*100 if checked_long else 0:.3f}% lỗi)")